# 12F-10 — Canonical dataset build

**Question:** How do trustworthy records form Program → Configuration → VDE → RUN → FuelCons tables?

This MVP exports a deterministic 75-program public-data slice from the canonical staging database. It does not rebuild production ETL.

In [ ]:
from pathlib import Path
import pandas as pd

search_roots = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(path for path in search_roots if (path / "AGENTS.md").exists() and (path / "etl").exists())
import sqlite3

CANONICAL_DB = REPO_ROOT / "etl/data/staging/sprint_12e2_epa_fuelcons/eco_drive_canonical_epa_fuelcons.db"
OUTPUT_DIR = REPO_ROOT / "notebooks/_data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
connection = sqlite3.connect(CANONICAL_DB.resolve().as_uri() + "?mode=ro", uri=True)


## Select a deterministic relational slice

In [ ]:
program_ids = pd.read_sql_query('''
SELECT DISTINCT vc.program_id
FROM vehicle_configuration vc
JOIN vde v ON v.vehicle_configuration_id=vc.vehicle_configuration_id
ORDER BY vc.program_id
LIMIT 75
''', connection)["program_id"].tolist()
placeholders = ",".join("?" for _ in program_ids)
program = pd.read_sql_query(f"SELECT * FROM program WHERE program_id IN ({placeholders})", connection, params=program_ids)
vehicle_configuration = pd.read_sql_query(f"SELECT * FROM vehicle_configuration WHERE program_id IN ({placeholders})", connection, params=program_ids)


In [ ]:
configuration_ids = vehicle_configuration["vehicle_configuration_id"].tolist()
config_marks = ",".join("?" for _ in configuration_ids)
vde = pd.read_sql_query(f"SELECT * FROM vde WHERE vehicle_configuration_id IN ({config_marks})", connection, params=configuration_ids)

vde_ids = vde["id"].tolist()
vde_marks = ",".join("?" for _ in vde_ids)
run = pd.read_sql_query(f"SELECT * FROM run WHERE vde_id IN ({vde_marks})", connection, params=vde_ids)
fuelcons = pd.read_sql_query(f"SELECT * FROM fuelcons WHERE vde_id IN ({vde_marks})", connection, params=vde_ids)

fuelcons_ids = fuelcons["id"].tolist()
if fuelcons_ids:
    fuelcons_marks = ",".join("?" for _ in fuelcons_ids)
    fuelcons_run_adoption = pd.read_sql_query(
        f'''SELECT * FROM fuelcons_run_adoption
            WHERE fuelcons_id IN ({fuelcons_marks})
            ORDER BY fuelcons_id, run_id, result_dimension''',
        connection,
        params=fuelcons_ids,
    )
else:
    fuelcons_run_adoption = pd.read_sql_query("SELECT * FROM fuelcons_run_adoption WHERE 0", connection)
connection.close()


## Check relationships before export

In [ ]:
assert set(vehicle_configuration["program_id"]) <= set(program["program_id"])
assert set(vde["vehicle_configuration_id"]) <= set(vehicle_configuration["vehicle_configuration_id"])
assert set(run["vde_id"]) <= set(vde["id"])
assert set(fuelcons["vde_id"]) <= set(vde["id"])
assert set(vde["vde_id_parent"].dropna().astype(int)) <= set(vde["id"])
assert set(fuelcons_run_adoption["fuelcons_id"]) <= set(fuelcons["id"])
assert set(fuelcons_run_adoption["run_id"]) <= set(run["run_id"])
assert set(fuelcons_run_adoption["vde_id"]) <= set(vde["id"])

fuelcons_vde = fuelcons.set_index("id")["vde_id"]
run_vde = run.set_index("run_id")["vde_id"]
assert fuelcons_run_adoption["vde_id"].eq(fuelcons_run_adoption["fuelcons_id"].map(fuelcons_vde)).all()
assert fuelcons_run_adoption["vde_id"].eq(fuelcons_run_adoption["run_id"].map(run_vde)).all()

tables = {"program": program, "vehicle_configuration": vehicle_configuration, "vde": vde, "run": run,
          "fuelcons": fuelcons, "fuelcons_run_adoption": fuelcons_run_adoption}
pd.Series({name: len(frame) for name, frame in tables.items()}, name="rows")


## Save explicit staging tables

In [ ]:
sort_keys = {"program": ["program_id"], "vehicle_configuration": ["vehicle_configuration_id"], "vde": ["id"],
             "run": ["run_id"], "fuelcons": ["id"],
             "fuelcons_run_adoption": ["fuelcons_id", "run_id", "result_dimension"]}
for name, frame in tables.items():
    path = OUTPUT_DIR / f"12f10_{name}.csv"
    frame.sort_values(sort_keys[name], kind="stable").to_csv(path, index=False)
    print(f"{name:24s} {len(frame):5d} rows -> {path.relative_to(REPO_ROOT)}")


## What did we learn?

Relational export order follows ownership: Program, Configuration, VDE, then RUN and FuelCons. Provenance and NULLs travel with the records; no notebook-only identity is invented.